# L3d Algorithm: Quicksort

So far, we've seen recursion in the context of mathematical functions, but recursion also solves algorithmic problems. Quicksort is the classic example: it orders an array by splitting it around one chosen element and then sorting the two smaller pieces the same way. Whether it terminates and how fast it runs both follow from how that split is done.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __State the recursive structure:__ Identify the base case and the recursive case of a divide-and-conquer sort. Explain why partitioning around a pivot leaves the two sides independent, so their sorted results can be concatenated rather than merged.
> * __Explain why the recursion terminates:__ Show that every recursive call receives a strictly smaller array once the placed pivot is excluded. Explain how a partition that keeps the pivot inside one of the sides can hand the same array back to the next level and recurse forever.
> * __Connect pivot choice to complexity:__ Explain why a pivot that splits the input evenly keeps the recursion shallow, which is what gives the algorithm its average-case running time. Explain why a pivot that is always the smallest or largest value removes only one element per level when the values are distinct, deepening the recursion until the work becomes quadratic.

Let's get started!
___

## The Algorithm
The split determines everything else, so let's be precise about what it produces before writing it down.

> __How does Quicksort work?__ Quicksort is a recursive sorting algorithm that works by selecting a pivot element and partitioning the remaining elements according to their value relative to the pivot. The algorithm then recursively sorts the partitions until they have fewer than two elements. The choice of pivot is critical for the algorithm's efficiency.

Let's look at some pseudocode to illustrate the basic idea of Quicksort.

__Initialization__: You are given an unsorted numerical array $\mathbf{x}\in \mathbb{R}^n$ of $n$ elements, and you want to sort it in ascending order.

__Base case__: If $n\leq{1}$, then the array is already sorted, and we can return it as is.

__Recursive case__: Otherwise, we proceed with the following steps:
1. Select a pivot element $x_{\star}\in\mathbf{x}$ from the array ([our implementation](src/Compute.jl) takes the last element).
2. Partition __all__ of $\mathbf{x}$ into three groups by comparison with the pivot:
   - Lower: $\mathbf{L} = \{x\in\mathbf{x} \mid x < x_{\star}\}$ (strictly less than the pivot)
   - Equal: $\mathbf{E} = \{x\in\mathbf{x} \mid x \not< x_{\star}\;\text{and}\;x_{\star} \not< x\}$ (every value the comparison cannot separate from the pivot, the pivot included)
   - Upper: $\mathbf{U} = \{x\in\mathbf{x} \mid x > x_{\star}\}$ (strictly greater than the pivot)
3. Recursively sort $\mathbf{L}$ and $\mathbf{U}$. $\mathbf{E}$ needs no sorting, because the comparison cannot separate any of its elements.
4. Concatenate the results: return $\texttt{sort}(\mathbf{L}) \;+\; \mathbf{E} \;+\; \texttt{sort}(\mathbf{U})$.

All three groups are defined by the comparison the implementation makes rather than by numeric $<$ and $=$, and for ordinary numbers the two readings agree. They differ for special values: [the `isless(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.isless), which our implementation uses, orders `-0.0` before `0.0` even though the two are numerically equal, and it cannot separate two `NaN` values even though those are never numerically equal.

The three-way split in step 2 is a choice, and it is worth saying which part of it the algorithm cannot do without.

> __Why three groups and not two?__
>
> Termination needs only one thing: the pivot must never appear in a subproblem again. With the pivot taken out, both sides are strictly smaller than the input, so the recursion reaches the base case. A partition that leaves the pivot on one side can hand the same array back to the next level and recurse forever.
>
> The third group is about speed on duplicates. A two-way split has to push every value tied with the pivot onto one side or the other, so an array full of equal values can still recurse deeply. Putting the tied values in $\mathbf{E}$, where they are placed and never looked at again, lets an all-equal array finish after one partition step.

The figure runs one partition step on $\mathbf{x} = \left[8,2,6,4,6\right]$, whose last element is the pivot. Neither six can be separated from the pivot by the comparison, so both land in $\mathbf{E}$ and are never looked at again, and the two sides that get sorted recursively are each strictly shorter than the input.

<div>
    <center>
        <img src="figs/Fig-Quicksort-Partition.svg" width="620" alt="One quicksort partition step on the array 8, 2, 6, 4, 6: the last element is the pivot, the values split into a lower group holding 2 and 4, an equal group holding both sixes, and an upper group holding 8, and the sorted lower group, the equal group, and the sorted upper group concatenate to 2, 4, 6, 6, 8"/>
    </center>
</div>

The work is in the recursive structure. Each recursive call breaks down the problem into smaller subproblems until we reach the base case, where the array is trivially sorted.

___

## Worst-case time complexity
__Quicksort__ is a _divide-and-conquer_ sorting algorithm, and its running time depends on how deep the recursion goes.

Across all the calls at one depth, the partition work is $\mathcal{O}(n)$. Splits that stay roughly even give $\mathcal{O}(\log n)$ levels, which is the $\mathcal{O}(n\log n)$ balanced case, and random input gives the same bound on average.

The worst case is a pivot that is always the smallest or the largest value in its subproblem. When the values are distinct, that removes one element per level, and since our implementation takes the last element, an already sorted array of distinct values, in either direction, gives $n$ levels and $\mathcal{O}(n^{2})$ work.

Depth costs memory as well as time. Every level allocates new lower, equal, and upper arrays on the heap, and the nested calls themselves hold stack frames, so a large sorted input can exhaust the call stack.

___

## Summary
Quicksort sorts by splitting an array around a pivot and sorting each side the same way, so both its termination and its running time are properties of the partition rather than of the recursion.

> __Key Takeaways:__
>
> * __Partitioning makes the subproblems independent:__ Once every element is on the correct side of the pivot, the two sides can be sorted without any further reference to each other. That independence is what lets the sorted results be concatenated rather than merged.
> * __Termination is a property of the partition, not the recursion:__ Excluding the placed pivot is what guarantees every recursive call receives a strictly smaller array. Grouping the tied values with it goes further, letting an array of equal values finish after a single partition pass.
> * __Pivot choice sets the complexity:__ Each partition pass costs work proportional to the size of its own subproblem. The difference between the average case and the quadratic worst case is therefore how many levels of subproblems the chosen pivot creates, and how large each of them is.

Pivot choice is what separates an $\mathcal{O}(n\log n)$ sort from one that falls back to the quadratic behavior it was meant to beat, which is why [the sorting lab](CHEME-5800-L3d-Lab-AnotherLookAtSorting-Fall-2026.ipynb) times this recursion on random data rather than on input chosen to make the last element extreme every time.
___